# SOC triage: alerts, phishing, threat intel and AI verdicts

Work a ten-alert morning queue at a lake-side hotel group: triage five reported emails, write and test a YARA rule, check indicators against intel and map them to ATT&CK, decode a suspicious command in CyberChef, and review an AI assistant's verdicts, including one that obeyed text an attacker wrote.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `ai-verdicts.json`, `alerts-wednesday.json`, `intel.json`, `known_bad.txt`, `mail.log`, `proxy.log`, `README.txt`, `reported/gst-refund.eml`, `reported/guest-complaint.eml`, `reported/linen-offer.eml`, `reported/payslips.eml`, `reported/supplier-bank-change.eml`, `suspect/boarding-pass.strings.txt`, `travel.csv`.

## 1. How the alert queue works

Before you triage, learn the program. An alert queue (the triage console of a SOC) takes alerts from many sources, the mail gateway, the proxy, antivirus, sign-in logs and staff who press **Report phishing**, and lines them up for a person to decide.

Every alert carries the same fields: an **ID**, the **time**, the **rule** that fired, the **host** and **user** it is about, a **detail** line, and the **evidence** files to open next. The queue doesn't decide anything. You give each alert one **disposition**:

| Disposition | Meaning |
|---|---|
| True positive: escalate | Real, and a person must act now |
| True positive: handled | Real, and already contained (blocked, removed, quarantined) |
| Benign: expected activity | The rule worked, but what it saw is normal here |
| False positive: rule fired wrongly | The rule matched something it was never meant to catch |
| Needs more information | You can't decide yet: write the question and who can answer it |

The Linux cell shows the raw queue file, so you can see that the program is only reading JSON. The queue below is the one you will work today, at **Lotus Lake Hotels** in Udaipur. Don't set any dispositions yet: just open it and look around. Read `README.txt` first.

- [ ] Compare the raw JSON with how the queue shows the same alert
- [ ] Open one alert's evidence list
- [ ] Find the summary line that counts dispositions

> Hint: The difference between "handled" and "benign" is whether anything bad happened. A blocked phishing click is handled; a scanner doing its job is benign.

```tool alerts
{"file": "alerts-wednesday.json", "d": {}, "n": 0, "blank": ["d"]}
```

?> Which field of an alert lists the files to open next? :: sha256:ee8250fb76e094b34b471f13a73dbbe51d1ae142e9df59d7c0d31ec20f0a0a8e
?> How many dispositions can you choose from? :: sha256:ef2d127de37b942baad06145e54b0c619a1f22327b2ebbcfbec78f5564afe39d

In [ ]:
%%bash
cd ~/case
head -c 700 alerts-wednesday.json; echo
grep -c '"id"' alerts-wednesday.json

## 2. Read the morning queue

It's Wednesday 11 November, the first busy week of the wedding season at Lotus Lake Hotels. Hotels are a favourite target at this time of year: bookings, advances to suppliers and guests' cards all move fast. You have ten alerts. Your job is to close what is noise, finish what is already handled, and escalate only what needs a person now.

Read the queue once end to end before you decide anything. Then hash the evidence, so you can show later that nothing changed while you worked.

- [ ] Hash every evidence file
- [ ] Read all ten alerts
- [ ] Which alerts are reported emails, and which come from machines?
- [ ] Which one has money and a deadline today?

> Hint: Look for money and a deadline. Attackers who can't get past the mail filter often go after the payment process instead.

?> How many alerts are in the Wednesday queue? :: sha256:4a44dc15364204a80fe80e9039455cc1608281820fe2b24f1e5233ade6af1dd5
?> At what time (IST) does today's payment run close? (HH:MM) :: sha256:fb4506210213175c1ef631304e7ee73094ed4cc322b2b02059f862514ccab225

In [ ]:
%%bash
cd ~/case
sha256sum *.* reported/* suspect/*
echo ---
grep -E '"(id|rule|host)"' alerts-wednesday.json

## 3. Triage the reported emails

Five emails were reported. For each one, read who **really** sent it (Return-Path and the Received line), what SPF, DKIM and DMARC said, where the link **really** goes, and which pressure tactics it uses. Then give a verdict: phishing, payment fraud (BEC), spam, or genuine.

- [ ] Sender, Reply-To and Return-Path for each email
- [ ] Authentication results, and which domain each result is about
- [ ] The real link target, not the link text
- [ ] Pressure tactics: urgency, secrecy, authority, fear of loss
- [ ] Anything in the email that isn't meant for a human reader
- [ ] A verdict for each

> Hint: In guest-complaint.eml the link text says extranet.bookingpartner.example, but the href goes somewhere else. In supplier-bank-change.eml the display name says General Manager, but the address is a free mail account, and SPF, DKIM and DMARC all pass for that free mail domain. gst-refund.eml has a line in 1-pixel white text.

```tool email
{"d": {"reported/guest-complaint.eml": "phish", "reported/gst-refund.eml": "phish", "reported/linen-offer.eml": "spam", "reported/payslips.eml": "legit", "reported/supplier-bank-change.eml": "bec"}, "n": 3, "files": ["reported/guest-complaint.eml", "reported/gst-refund.eml", "reported/linen-offer.eml", "reported/payslips.eml", "reported/supplier-bank-change.eml"], "blank": ["d"]}
```

?> Which host does the guest-complaint link really go to? :: sha256:38da468b2f77c323d6bc622494a5afb080fa3894f8dec28b7016c6770d6a40c5
?> How many of the five emails are phishing or payment fraud? :: sha256:4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce

In [ ]:
%%bash
cd ~/case/reported
grep -H -E '^(From|Reply-To|Return-Path):' *.eml
echo ---
grep -H -o 'spf=[a-z]*\|dkim=[a-z]*\|dmarc=[a-z]*' *.eml
echo ---
grep -H -o 'href="[^"]*"' *.eml
echo ---
grep -H 'font-size:1px' *.eml

## 4. Work the queue

Now give all ten alerts a disposition and a one-line reason. **Exactly one** should be escalated: the one where a person must act in the next few hours. Already-contained activity is "true positive: handled". Normal activity that a rule noticed is "benign". A rule that fired on something it was never meant to catch is "false positive".

- [ ] Every alert has a disposition
- [ ] One escalation
- [ ] Each reason points at evidence (a log line, a header, a file)
- [ ] Write the notes for the next shift

> Hint: AL-4106 runs every night at 21:30 UTC under an IT service account. Decode the command in the next step before you decide. For AL-4109, open travel.csv.

```tool alerts
{"file": "alerts-wednesday.json", "d": {"AL-4101": "handled", "AL-4102": "escalate", "AL-4103": "handled", "AL-4104": "benign", "AL-4105": "benign", "AL-4106": "false", "AL-4107": "handled", "AL-4108": "benign", "AL-4109": "benign", "AL-4110": "handled"}, "n": 1, "blank": ["d"]}
```

?> Which alert is the one to escalate? :: sha256:6d1e2fcf9800ff125596da76593194984ba52da8529fc24aa11e2a46c66f02e3

## 5. Decode the suspicious command

AL-4106 is an encoded PowerShell command on pms01, the server that runs the hotel's booking system. Encoding hides what a command does, which is why attackers use it, but IT scripts sometimes use it too. Decode it before you judge it.

[Open the command in CyberChef](https://zeffiroh.github.io/ap-cybersecurity/cyberchef/#recipe=From_Base64%28%27A-Za-z0-9%252B/%253D%27,true,false%29Decode_text%28%27UTF-16LE%2520%281200%29%27%29&input=UXdCdkFHMEFjQUJ5QUdVQWN3QnpBQzBBUVFCeUFHTUFhQUJwQUhZQVpRQWdBQzBBVUFCaEFIUUFhQUFnQUVRQU9nQmNBRkFBVFFCVEFGd0FSUUI0QUhBQWJ3QnlBSFFBWEFBcUFDQUFMUUJFQUdVQWN3QjBBR2tBYmdCaEFIUUFhUUJ2QUc0QVVBQmhBSFFBYUFBZ0FGd0FYQUJ1QUdFQWN3QXdBREVBWEFCd0FHMEFjd0F0QUdJQVlRQmpBR3NBZFFCd0FGd0FjQUJ0QUhNQUxRQWtBQ2dBUndCbEFIUUFMUUJFQUdFQWRBQmxBQ0FBTFFCR0FHOEFjZ0J0QUdFQWRBQWdBSGtBZVFCNUFIa0FUUUJOQUdRQVpBQXBBQzRBZWdCcEFIQUFJQUF0QUVZQWJ3QnlBR01BWlFBPQ==) (From Base64, then Decode text as UTF-16LE). Or decode it in the Linux cell.

- [ ] Decode the command
- [ ] What does it do? Is anything in it harmful?
- [ ] Which account runs it, and when?
- [ ] Write the disposition and a note to IT: an encoded command in a scheduled task trips the alert every night

> Hint: PowerShell's -EncodedCommand is Base64 of UTF-16LE text. On Linux, `base64 -d` decodes it and `tr -d '\000'` drops the zero bytes UTF-16 puts between letters.

?> Which PowerShell command does AL-4106 run? (the cmdlet name) :: sha256:fdb115f5e63e6ad53e6ddce05f1e8503d5798cc663ead83814ebd48fa52f90cc

In [ ]:
%%bash
echo 'QwBvAG0AcAByAGUAcwBzAC0AQQByAGMAaABpAHYAZQAgAC0AUABhAHQAaAAgAEQAOgBcAFAATQBTAFwARQB4AHAAbwByAHQAXAAqACAALQBEAGUAcwB0AGkAbgBhAHQAaQBvAG4AUABhAHQAaAAgAFwAXABuAGEAcwAwADEAXABwAG0AcwAtAGIAYQBjAGsAdQBwAFwAcABtAHMALQAkACgARwBlAHQALQBEAGEAdABlACAALQBGAG8AcgBtAGEAdAAgAHkAeQB5AHkATQBNAGQAZAApAC4AegBpAHAAIAAtAEYAbwByAGMAZQA=' | base64 -d | tr -d '\000'; echo

## 6. A YARA rule for the lure

Write a YARA rule that finds the guest-complaint lure in mail, so the mail gateway can catch the next one. Test it on **every** file. The first rule below matches on the phishing domain alone: see what else it catches. The tuned rule needs three things that only the lure email has.

- [ ] Test the noisy rule: which files does it match? Are they emails?
- [ ] Test the tuned rule: it should match only guest-complaint.eml
- [ ] Test the third rule on the quarantined file's strings
- [ ] Write one line on what each rule would miss

> Hint: The intel feed, the block list, the alert queue and the proxy log all mention the domain too. A rule for email should need things only the email has: the subject, the link path and the threat.

```tool yara
{"rule": "rule llh_phishing_domain\n{\n  meta:\n    description = \"Anything that mentions the phishing domain\"\n  strings:\n    $d = \"login-verify.example\"\n  condition:\n    $d\n}\n", "n": 5, "files": ["alerts-wednesday.json", "ai-verdicts.json", "intel.json", "known_bad.txt", "mail.log", "proxy.log", "reported/guest-complaint.eml"], "blank": ["rule"]}
```

```tool yara
{"rule": "rule llh_guest_complaint_lure\n{\n  meta:\n    description = \"The fake guest-complaint email aimed at hotel reservations teams\"\n    author = \"SOC\"\n    reference = \"AL-4101, message 7C1A20F11\"\n  strings:\n    $subject = \"Subject: Guest complaint BKG-\"\n    $link = \"login-verify.example/complaints/\"\n    $press = \"hidden from search results\"\n  condition:\n    all of them\n}\n", "n": 1, "files": ["reported/guest-complaint.eml"], "blank": ["rule"]}
```

```tool yara
{"rule": "rule bp_grab_stealer_strings\n{\n  meta:\n    description = \"Strings from Boarding_Pass.pdf.exe, a password stealer left on guest USB sticks\"\n  strings:\n    $mutex = \"Global\\\\bp-grab-7731\"\n    $chrome = \"\\\\Google\\\\Chrome\\\\User Data\\\\Default\\\\Login Data\"\n    $flag = \"--grab-all\"\n    $exfil = \"telegram-relay.example/bot\"\n  condition:\n    3 of them\n}\n", "n": 1, "files": ["suspect/boarding-pass.strings.txt"], "blank": ["rule"]}
```

?> How many files does the domain-only rule match? :: sha256:ef2d127de37b942baad06145e54b0c619a1f22327b2ebbcfbec78f5564afe39d
?> How many files does the tuned lure rule match? :: sha256:6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b

## 7. Indicators and intel

Pull every indicator out of the evidence, check it against the intel feed, and map what you saw to ATT&CK. Your intel note lists each indicator **defanged** (login-verify[.]example), says which source knows it, and says what to do: block, watch or ignore.

- [ ] Extract indicators
- [ ] Which are on the feed, and from which source?
- [ ] Which names and addresses must not be blocked?
- [ ] Map the three campaigns to ATT&CK: the fake booking partner, the fake GST refund and the USB stealer, plus the payment fraud
- [ ] Write the intel note

> Hint: The feed also lists the real booking-partner extranet and internal systems as benign. Blocking extranet.bookingpartner.example would stop Reservations from working.

```tool ioc
{"attack": ["T1566.002", "T1583.001", "T1656", "T1091", "T1204.002", "T1555.003", "T1567"], "blank": ["iocs", "attack"]}
```

?> Which source on the feed reported bookingpartner-extranet.login-verify.example? :: sha256:491cb6a3a8d2f867112969b0c5996f30ea645dfce8fdd0332b102509a728b749

## 8. Review the AI assistant's verdicts

The SOC's AI triage assistant gave a verdict on every alert. It is fast and usually right, but it only sees the alert text, not your evidence, and it reads **everything** in that text, including words an attacker wrote. Agree, or override it and say why. **Every override needs a fact** from the evidence.

- [ ] Read each verdict and its reason
- [ ] Agree or override
- [ ] For each override, the fact that proves it wrong
- [ ] For each override, the kind of mistake: too trusting, too alarmed, missing context, or obeying text an attacker wrote

> Hint: Four verdicts are wrong. One trusted a display name. One believed a sentence inside the email that was written for it. One was alarmed by encoding without decoding it. One didn't know about the trade show.

```tool aisec
{"file": "ai-verdicts.json", "marks": {"AL-4101": "agree", "AL-4102": "override", "AL-4103": "override", "AL-4104": "agree", "AL-4105": "agree", "AL-4106": "override", "AL-4107": "agree", "AL-4108": "agree", "AL-4109": "override", "AL-4110": "agree"}, "n": 10, "blank": ["marks"]}
```

?> How many AI verdicts did you override? :: sha256:4b227777d4dd1fc61c6f884f48641d02b4d121d3fd328cb08b5531fcacdabf8a

## 9. Escalate the payment fraud

Write the one escalation. The person reading it is the Finance Controller, Pooja Mathur, who must stop a payment before 2 pm IST. Put the ask first, and say what you have **not** done (you didn't reply to the sender).

- [ ] The ask, with a time in the reader's time zone and in UTC
- [ ] Why it is high
- [ ] Know and don't know
- [ ] Where the evidence is

> Hint: A call on a number you already have is the fastest way to confirm. Never use contact details from the suspicious email.

```tool report
{"tpl": "escalation", "title": "Escalation: payment fraud attempt against Accounts", "text": "# Escalation: payment fraud attempt against Accounts\n\n## The ask\n\nPlease take the Rs 12,60,000 advance to Rang Mahal Decorators out of today's payment run, which closes at 14:00 IST (08:30 UTC). Finance Controller, please call Vikram Shekhawat on his known mobile number to confirm, and call Rang Mahal Decorators on the number in the supplier master file, not the one in the email.\n\n## Severity and why\n\nHigh: a payment is due today, the bank details in the request are new, and the request came from outside the company while pretending to be the General Manager.\n\n## What we know\n\n- Tue 10 Nov 12:05 UTC the email arrived for r.jain; he forwarded it to the SOC on Wed 11 Nov 04:20 UTC (AL-4102).\n- It is signed \"Vikram Shekhawat (General Manager)\" but sent from vikram.gm.lotuslake@freemail.example.\n- It asks for the supplier's bank details to be changed and the advance released by 2 pm, says to keep it secret, and says to reply only to the freemail address.\n- SPF, DKIM and DMARC all pass for freemail.example: it really came from a free mail account, not from lotuslakehotels.com. Passing proves who sent it, not that it is safe.\n\n## What we don't know\n\n- Whether the General Manager knows about the request (he has not been reached yet).\n- Whether the supplier's details were already changed in the accounts system, and whether anyone else in Accounts got the same email.\n\n## Impact so far\n\nOne accounts user and one pending payment are at risk. No account or computer is known to be compromised.\n\n## What I've done\n\nNothing has been changed. I have not replied to the sender.\n\n## Evidence\n\nreported/supplier-bank-change.eml is saved in the case folder with its SHA-256 hash; the queue with dispositions is alerts-wednesday.json.\n", "n": 7, "blank": ["text"]}
```

## 10. Hand in

Hand in this notebook. It carries your dispositions, email verdicts, YARA tests, intel note, AI review and escalation.

**Rubric**

| Criterion | Points |
|---|---|
| Ten dispositions, each with a one-line reason a stranger could check | 6 |
| Exactly one escalation, written with the ask first | 3 |
| Phishing report: sender, authentication results, the real link, pressure tactics and a verdict for each email | 5 |
| YARA rule that matches the lure and not the intel feed or logs, with its test result | 4 |
| Intel note: indicators checked against the feed, defanged, mapped to ATT&CK | 4 |
| AI-verdict review log: every verdict agreed or overridden, each override backed by evidence, including the alert that carried text written for the AI | 4 |
| Plain writing: facts kept apart from guesses | 2 |

- [ ] All questions answered
- [ ] Ten dispositions, one escalation
- [ ] Each YARA rule tested
- [ ] Each AI override has a fact
- [ ] Press Hand in